In [2]:
!pip install pyspark

In [19]:
!rm -f yellow_tripdata_2024-10.parquet
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet"

--2025-03-03 19:16:30--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 65.8.245.51, 65.8.245.178, 65.8.245.171, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|65.8.245.51|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M   150MB/s    in 0.4s    

2025-03-03 19:16:30 (150 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [20]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('TaxiData') \
    .getOrCreate()

df = spark.read.parquet("yellow_tripdata_2024-10.parquet")
df.printSchema()  # Check the schema
df.show(5)        # Show first 5 rows

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------

In [6]:
# df.write.parquet('zones')

In [7]:
spark.version

'3.5.5'

In [28]:
new_df =df.repartition(4)
new_df.write.parquet('df_parquet', mode='overwite')

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/content/df_parquet already exists. Set mode as "overwrite" to overwrite the existing path.

In [29]:
!du -sh df_parquet/  # Total size of the folder
!ls -lh df_parquet/  # List individual file sizes

93M	df_parquet/
total 93M
-rw-r--r-- 1 root root 24M Mar  3 19:19 part-00000-3acf4fd6-11ae-44c5-8bf4-2f6bbd7af8fe-c000.snappy.parquet
-rw-r--r-- 1 root root 24M Mar  3 19:19 part-00001-3acf4fd6-11ae-44c5-8bf4-2f6bbd7af8fe-c000.snappy.parquet
-rw-r--r-- 1 root root 24M Mar  3 19:19 part-00002-3acf4fd6-11ae-44c5-8bf4-2f6bbd7af8fe-c000.snappy.parquet
-rw-r--r-- 1 root root 24M Mar  3 19:19 part-00003-3acf4fd6-11ae-44c5-8bf4-2f6bbd7af8fe-c000.snappy.parquet
-rw-r--r-- 1 root root   0 Mar  3 19:19 _SUCCESS


In [35]:
# prompt: How many taxi trips were there on the 15th of October?
# Consider only trips that started on the 15th of October.

from pyspark.sql.functions import to_date, col

# Assuming 'tpep_pickup_datetime' is the column representing the pickup time
trips_on_15th = df.filter(to_date(col("tpep_pickup_datetime")) == "2024-10-15").count()

print(f"Number of taxi trips on October 15th: {trips_on_15th}")


Number of taxi trips on October 15th: 128893


In [34]:
from pyspark.sql.functions import col, unix_timestamp

# Create a new column for trip duration in hours
df = df.withColumn(
    "trip_hours",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600
)

# Find the maximum trip duration
longest_trip = df.selectExpr("MAX(trip_hours) as max_trip_hours").collect()[0]["max_trip_hours"]

print(f"Longest trip duration: {longest_trip:.2f} hours")


Longest trip duration: 162.62 hours


In [36]:
!apt-get install -y net-tools  # Install net-tools to check open ports
!netstat -tulnp | grep 4040

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  net-tools
0 upgraded, 1 newly installed, 0 to remove and 29 not upgraded.
Need to get 204 kB of archives.
After this operation, 819 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 net-tools amd64 1.60+git20181103.0eebece-1ubuntu5 [204 kB]
Fetched 204 kB in 1s (333 kB/s)
Selecting previously unselected package net-tools.
(Reading database ... 124947 files and directories currently installed.)
Preparing to unpack .../net-tools_1.60+git20181103.0eebece-1ubuntu5_amd64.deb ...
Unpacking net-tools (1.60+git20181103.0eebece-1ubuntu5) ...
Setting up net-tools (1.60+git20181103.0eebece-1ubuntu5) ...
Processing triggers for man-db (2.10.2-1) ...
tcp        0      0 0.0.0.0:4040            0.0.0.0:*               LISTEN      954/java            


In [37]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(4040)"))

https://5ssunr4b93r-496ff2e9c6d22116-4040-colab.googleusercontent.com/


In [38]:
spark = SparkSession.builder.config("spark.ui.port", "4050").getOrCreate()


In [40]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-03 20:26:16--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 65.8.245.51, 65.8.245.171, 65.8.245.178, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|65.8.245.51|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-03 20:26:16 (265 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [41]:
lookup = spark.read.csv("taxi_zone_lookup.csv", header=True)
lookup.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [42]:
joined_df = df.join(lookup, df.PULocationID == lookup.LocationID, "left")
joined_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------+----------+---------+-------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|          trip_hours|LocationID|  Borough|               Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+--------------------+----------+------

In [46]:
joined_df.createOrReplaceTempView ('temp')

In [54]:
spark.sql("""
    SELECT Zone, COUNT(1) as count
    FROM temp
    GROUP BY Zone
    ORDER BY count ASC
    LIMIT 1
""").show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
+--------------------+-----+

